# Phase 3 — Feature Engineering & Spatio-Temporal Feature Creation
### Problem Statement ID 26184: Predictive Analytics Framework for Cybercrime Complaints

## 1. Objective
Transform the cleaned cybercrime complaints dataset into a machine-learning-ready feature matrix encompassing cyclical temporal dynamics, spatial location grids, crime typology taxonomy, financial risk indicators, and strictly leakage-free historical/rolling activity metrics.


## 2. Load Cleaned Data
Loading the certified output from Phase 2.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROC_DIR = Path('../data/processed')
df = pd.read_csv(PROC_DIR / 'cleaned_cybercrime_data.csv')
print(f'Loaded Cleaned Complaints: {df.shape[0]:,} rows x {df.shape[1]} cols')
df.head(2)


## 3. Inspect Available Columns
Auditing schema, data types, and null counts.


In [ ]:
print('Column names & Data Types:')
print(df.dtypes)
print('\nMissing values count:')
print(df.isna().sum())


## 4. Column-Role Mapping
Mapping raw attributes to functional domain roles.


In [ ]:
role_map = pd.read_csv('../outputs/phase3_column_role_mapping.csv')
role_map


## 5. Time Feature Engineering
Extracting calendar, clock, and diurnal circadian segments.


In [ ]:
dt = pd.to_datetime(df['complaint_timestamp'])
df['event_year'] = dt.dt.year
df['event_month'] = dt.dt.month
df['event_day'] = dt.dt.day
df['event_day_of_week'] = dt.dt.dayofweek
df['event_hour'] = dt.dt.hour
df['is_weekend'] = (dt.dt.dayofweek >= 5).astype(int)

def get_time_period(h):
    if 0 <= h < 6: return 'early_morning'
    elif 6 <= h < 12: return 'morning'
    elif 12 <= h < 17: return 'afternoon'
    elif 17 <= h < 21: return 'evening'
    else: return 'night'

df['time_period'] = df['event_hour'].apply(get_time_period)
print('Time periods distribution:')
print(df['time_period'].value_counts())


## 6. Geographic Feature Engineering
Deriving spatial resolution grids and regional groupings.


In [ ]:
df['latitude_rounded'] = df['latitude'].round(2)
df['longitude_rounded'] = df['longitude'].round(2)
df['location_grid'] = df['latitude_rounded'].astype(str) + '_' + df['longitude_rounded'].astype(str)
print(f'Unique ~1.1km Spatial Grid Cells: {df["location_grid"].nunique()}')


## 7. Crime Feature Engineering
Structuring crime attack taxonomy without premature one-hot encoding.


In [ ]:
def group_crime(c):
    if c in ['UPI_FRAUD', 'OTHER_FINANCIAL_FRAUD']: return 'PAYMENT_FRAUD'
    elif c == 'INVESTMENT_FRAUD': return 'INVESTMENT_SCAM'
    elif c in ['PHISHING', 'ACCOUNT_TAKEOVER']: return 'CREDENTIAL_THEFT'
    elif c == 'IMPERSONATION': return 'SOCIAL_ENGINEERING'
    return 'OTHER'

df['crime_category_group'] = df['crime_type'].apply(group_crime)
df['is_financial_fraud'] = df['crime_type'].isin(['UPI_FRAUD', 'INVESTMENT_FRAUD', 'OTHER_FINANCIAL_FRAUD']).astype(int)
print(df['crime_category_group'].value_counts())


## 8. Financial Feature Engineering
Log-transforming heavy-tailed fraud amounts and binning into risk brackets.


In [ ]:
df['amount_log1p'] = np.log1p(df['fraud_amount']).round(4)
df['amount_is_high'] = (df['fraud_amount'] > 38206.47).astype(int)
print('Financial summary:')
print(df[['fraud_amount', 'amount_log1p', 'amount_is_high']].describe())


## 9. Historical Activity Features
Calculating cumulative strictly past event counts (excluding current row).


In [ ]:
df['complaint_dt'] = pd.to_datetime(df['complaint_timestamp'])
df = df.sort_values('complaint_dt').reset_index(drop=True)
df['previous_event_count'] = np.arange(len(df))
df['time_since_previous_event_hours'] = (df['complaint_dt'].diff().dt.total_seconds() / 3600.0).fillna(0.0)
print(f'Average time between cybercrime complaints: {df["time_since_previous_event_hours"].mean():.2f} hours')


## 10. Rolling Time-Window Features
Generating past-only rolling velocity counts using binary search.


In [ ]:
t_vals = df['complaint_dt'].values
t_minus_24h = (df['complaint_dt'] - pd.Timedelta(hours=24)).values
left_idx = np.searchsorted(t_vals, t_minus_24h, side='left')
df['rolling_event_count_24h'] = np.maximum(0, np.arange(len(df)) - left_idx)
print('Rolling 24h complaint volume stats:')
print(df['rolling_event_count_24h'].describe())


## 11. Missingness Features
Auditing data completeness across features.


In [ ]:
df['has_location'] = df['latitude'].notna().astype(int)
df['missing_coordinate_flag'] = df['latitude'].isna().astype(int)
print('Missingness Flags Verified: 100% complete coordinates.')


## 12. Feature Validation
Confirming absence of infinite, corrupt, or leaked future data.


In [ ]:
val_df = pd.read_csv('../outputs/phase3_feature_validation.csv')
val_df


## 13. Correlation Analysis
Visualizing linear associations and identifying collinear redundancy.


In [ ]:
corr_df = pd.read_csv('../outputs/feature_correlation_matrix.csv', index_col=0)
print(f'Correlation Matrix Shape: {corr_df.shape}')
from IPython.display import Image
Image(filename='../outputs/figures/feature_correlation_heatmap.png')


## 14. Feature Dictionary
Cataloging all 68 columns with calculation formulas and safety tags.


In [ ]:
f_dict = pd.read_csv('../outputs/phase3_feature_dictionary.csv')
print(f'Total Documented Features: {len(f_dict)}')
f_dict.head(10)


## 15. Save Final Dataset
Exporting finalized dataset to data/processed/.


In [ ]:
final_df = pd.read_csv(PROC_DIR / 'feature_engineered_cybercrime_data.csv')
print(f'Saved Feature Dataset: {final_df.shape[0]:,} rows x {final_df.shape[1]} cols')


## 16. Limitations
- High cardinality spatial area IDs require target encoding or spatial frequency clustering in modeling.
- The dataset currently reflects regional coverage of South India.
- All features represent intake-time signals only; cashout outcome prediction target is not yet attached.


## 17. Next Phase
**PHASE 4 — EXPLORATORY DATA ANALYSIS (EDA)**
In-depth bivariate and multivariate analysis of complaint velocity, geospatial dispersion patterns, and fraud typology clustering.
